# Task 1 — Data Enrichment
**Ethiopia Financial Inclusion Forecast**

This notebook reproducibly builds the enriched dataset from the starter dataset. Every
added record is documented in detail in `../data_enrichment_log.md` (source URL, exact
quote, confidence, collector, date, and rationale) — this notebook shows the mechanics of
*how* those records were added and validated against the schema, not a repeat of the
source-by-source narrative.

**Summary: 17 new records added on top of the 57 in the starter dataset -> 74 total.**

| Addition | Count | Fills |
|---|---|---|
| Observations | 9 | 2020 ACCESS baseline, 2024 gender split, mobile subscriptions, and — most importantly — the previously-missing `USG_DIGITAL_PAYMENT` Findex Usage target |
| Targets | 1 | Explicit NFIS-II gender-gap target (10pp by 2025) |
| Events | 3 | SBB/91/2024 gender governance directive, NDPS Phase One & Two |
| Impact links | 5 | Fills the zero-impact-link gap for NFIS-II; adds theory-based links for the newest events |


In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from data_loader import load_unified_data, load_reference_codes, validate_against_reference, get_events, get_impact_links

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

starter = load_unified_data('../data/raw/ethiopia_fi_unified_data.csv')
enriched = load_unified_data('../data/processed/ethiopia_fi_unified_data_enriched.csv')
ref = load_reference_codes('../data/raw/reference_codes.csv')

print(f"Starter dataset:  {starter.shape[0]} records")
print(f"Enriched dataset: {enriched.shape[0]} records  (+{enriched.shape[0]-starter.shape[0]})")

Starter dataset:  57 records
Enriched dataset: 74 records  (+17)


## 1. Schema Compliance Check

Every new record must use only values that exist in `reference_codes.csv`, and events must never carry a pre-assigned pillar.

In [2]:
issues = validate_against_reference(enriched, ref)
print("Values outside reference_codes.csv:", issues if issues else "None (fully compliant)")

new_events = get_events(enriched)
print()
print("Events with a non-empty pillar (must be 0):", new_events['pillar'].notna().sum())

impacts = get_impact_links(enriched)
events_ids = set(new_events['record_id'])
orphans = set(impacts['parent_id'].dropna()) - events_ids
print("impact_link rows whose parent_id doesn't match any event (must be empty):", orphans)

Values outside reference_codes.csv: None (fully compliant)

Events with a non-empty pillar (must be 0): 0
impact_link rows whose parent_id doesn't match any event (must be empty): set()


## 2. What Changed, By record_type

In [3]:
comparison = pd.DataFrame({
    'starter': starter['record_type'].value_counts(),
    'enriched': enriched['record_type'].value_counts(),
}).fillna(0).astype(int)
comparison['added'] = comparison['enriched'] - comparison['starter']
comparison

,starter,enriched,added
record_type,,,
observation,30,38,8
impact_link,14,19,5
event,10,13,3
target,3,4,1


## 3. New Indicators Introduced

Three new `indicator_code` values were introduced (indicator_code is not a schema-controlled field, so this is a valid extension, not a violation).

In [4]:
new_ids = ['REC_0034','REC_0035','REC_0036','REC_0037','REC_0038','REC_0039','REC_0040','REC_0041','REC_0042',
           'EVT_0011','EVT_0012','EVT_0013','IMP_0015','IMP_0016','IMP_0017','IMP_0018','IMP_0019']
new_records = enriched[enriched['record_id'].isin(new_ids)]

new_indicators = new_records.loc[new_records['record_type'].isin(['observation','target']), 'indicator_code'].unique()
print("New indicator codes:", list(new_indicators))
print()
existing_indicators = starter.loc[starter['record_type'].isin(['observation','target']), 'indicator_code'].unique()
print("Confirmed not present in starter dataset:",
      [i for i in new_indicators if i not in existing_indicators])

New indicator codes: ['ACC_OWNERSHIP', 'ACC_MOBILE_SUBS', 'USG_DIGITAL_PAYMENT', 'USG_WAGE_DIGITAL', 'GEN_GAP_ACC']

Confirmed not present in starter dataset: ['ACC_MOBILE_SUBS', 'USG_DIGITAL_PAYMENT', 'USG_WAGE_DIGITAL']


## 4. The Most Important Addition: `USG_DIGITAL_PAYMENT`

This is the Findex "digital payment adoption" metric — one of the two headline forecasting targets named in the project brief — which was entirely absent from the starter dataset.

In [5]:
dp = enriched[enriched['indicator_code'] == 'USG_DIGITAL_PAYMENT'][
    ['record_id','observation_date','value_numeric','source_name','confidence']
].sort_values('observation_date')
dp

,record_id,observation_date,value_numeric,source_name,confidence
62,REC_0039,2021-12-31,20.0,World Bank Africa Can End Poverty blog (Global...,medium
63,REC_0040,2024-11-29,35.0,"Global Findex Database 2024, as cited in proje...",medium


## 5. Filling the NFIS-II Impact-Link Gap

`01.ipynb` flagged `EVT_0009` (NFIS-II Strategy Launch) as having zero impact_links in the starter data. It now has two — one for its headline ACCESS target and one for its explicit GENDER-gap target.

In [6]:
nfis_impacts = enriched[enriched['parent_id'] == 'EVT_0009'][
    ['record_id','pillar','related_indicator','impact_direction','impact_magnitude','impact_estimate','evidence_basis']
]
nfis_impacts

,record_id,pillar,related_indicator,impact_direction,impact_magnitude,impact_estimate,evidence_basis
70,IMP_0016,ACCESS,ACC_OWNERSHIP,increase,high,25.0,empirical
71,IMP_0017,GENDER,GEN_GAP_ACC,decrease,medium,-9.0,empirical


## 6. Full New-Record Listing

For the complete source URL / exact quote / confidence / rationale behind each row below, see `../data_enrichment_log.md`.

In [7]:
new_records[['record_id','record_type','category','pillar','indicator','indicator_code',
             'value_numeric','observation_date','source_type','confidence']]

,record_id,record_type,category,pillar,indicator,indicator_code,value_numeric,observation_date,source_type,confidence
57,REC_0034,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,4.500000e+01,2020-12-31,policy,high
58,REC_0035,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,5.653000e+01,2024-11-29,research,medium
59,REC_0036,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,4.162000e+01,2024-11-29,research,medium
60,REC_0037,observation,NaN,ACCESS,Mobile Subscriptions (count),ACC_MOBILE_SUBS,7.410000e+07,2020-06-30,research,medium
61,REC_0038,observation,NaN,ACCESS,Mobile Subscriptions (count),ACC_MOBILE_SUBS,2.694000e+08,2024-06-30,research,medium
62,REC_0039,observation,NaN,USAGE,Digital Payment Adoption Rate,USG_DIGITAL_PAYMENT,2.000000e+01,2021-12-31,research,medium
63,REC_0040,observation,NaN,USAGE,Digital Payment Adoption Rate,USG_DIGITAL_PAYMENT,3.500000e+01,2024-11-29,survey,medium
64,REC_0041,observation,NaN,USAGE,Used Account to Receive Wages,USG_WAGE_DIGITAL,1.500000e+01,2024-11-29,survey,medium
65,REC_0042,target,NaN,GENDER,Account Ownership Gender Gap,GEN_GAP_ACC,1.000000e+01,2025-12-31,policy,high
66,EVT_0011,event,regulation,NaN,Bank Corporate Governance Directive (SBB/91/20...,NaN,NaN,2024-06-12,regulator,high


## 7. Save & Next Steps

The enriched dataset is saved at `../data/processed/ethiopia_fi_unified_data_enriched.csv`
and is the file all subsequent tasks (EDA, impact modeling, forecasting, dashboard) should
load. Known remaining gaps (QUALITY/DEPTH/TRUST pillars, urban/rural splits, the 2011
Findex-inclusion discrepancy) are documented at the end of `data_enrichment_log.md` rather
than filled with unverified figures.
